# CSE152B - Spring 2022: Homework 2

## Question 1: Warm Up

We will first try SFM using the original implementation from [$\mathtt{libviso2}$](http://www.cvlibs.net/software/libviso/) (we wrapped it with Python wrappers at ``pyviso/src``). We will test on a dataset containing 300 images from one sequence of the KITTI dataset with ground-truth camera poses and camera calibration information. 

Run the SFM algorithm using the following script. You are required to report two error metrics. The error metric for rotation is defined as the mean of Frobenius norm of the difference between the ground-truth rotation matrix and predicted rotation matrix. The error metric for translation is defined as mean of the L2 distance. Both errors will be printed on the screen as you run the code.  **(5 points)**

For Q1: Toggling ``if_vis = True/False`` allows you to enable/disable the visualization. Disabling the visualization will make the for loop run significantly faster but will not visualize the correspondence/flow and trajectory.

In [2]:
# RUN THIS ONLY ONCE at each start of the notebook
import os
import sys
# change your base path
# os.chdir('./pyviso/') # './'

import viso2

In [ ]:
import os
import numpy as np
np.set_printoptions(precision=4)
np.set_printoptions(suppress=True)
import viso2
import matplotlib.pyplot as plt
from skimage.io import imread
import time

def errorMetric(RPred, RGt, TPred, TGt):
    diffRot = (RPred - RGt)
    diffTrans = (TPred - TGt)
    errorRot = np.sqrt(np.sum(np.multiply(diffRot.reshape(-1), diffRot.reshape(-1))))
    errorTrans = np.sqrt(np.sum(np.multiply(diffTrans.reshape(-1), diffTrans.reshape(-1))))

    return errorRot, errorTrans

# set to True to do the visualization per frame; the images will be saved at '.vis/'. \\
# Turn off if you just want the camera poses and errors
if_vis = True 

# if True the visualization per frame is going to be displayed realtime on screen\\
# (only work on a local machine with an active X graphics server); \\
# if False there will be no real-time display, but in both options the images will be saved
if_on_screen = False 

# parameter settings (for an example, please download

dataset_path = 'dataset_SfM' # On the cluster

img_dir      = os.path.join(dataset_path, 'sequences/00/image_0')
gt_dir       = os.path.join(dataset_path, 'poses/00.txt')
calibFile    = os.path.join(dataset_path, 'sequences/00/calib.txt')
border       = 50;
gap          = 15;

# Load the camera calibration information
with open(calibFile) as fid:
    calibLines = fid.readlines()
    calibLines = [calibLine.strip() for calibLine in calibLines]

calibInfo = [float(calibStr) for calibStr in calibLines[0].split(' ')[1:]]
# param = {'f': calibInfo[0], 'cu': calibInfo[2], 'cv': calibInfo[6]}

# Load the ground-truth depth and rotation
with open(gt_dir) as fid:
    gtTr = [[float(TrStr) for TrStr in line.strip().split(' ')] for line in fid.readlines()]
gtTr = np.asarray(gtTr).reshape(-1, 3, 4)

# param['height'] = 1.6
# param['pitch']  = -0.08
# param['match'] = {'pre_step_size': 64}
first_frame  = 0
last_frame   = 300
epi = 1e-8

# init visual odometry
params = viso2.Mono_parameters()
params.calib.f = calibInfo[0]
params.calib.cu = calibInfo[2]
params.calib.cv = calibInfo[6]
params.height = 1.6
params.pitch = -0.08


first_frame  = 0
last_frame   = 300

# init transformation matrix array
Tr_total = []
Tr_total_np = []
Tr_total.append(viso2.Matrix.eye(4))
Tr_total_np.append(np.eye(4))

# init viso module
visoMono = viso2.VisualOdometryMono(params)

if if_vis:
    save_path = 'vis'
    os.makedirs(save_path, exist_ok=True)

    # create figure that is currently EMPTY
    fig = plt.figure(figsize=(10, 15))
    ax1 = plt.subplot(211)
    ax1.axis('off')
    ax2 = plt.subplot(212)
    ax2.set_xticks(np.arange(-100, 100, step=10))
    ax2.set_yticks(np.arange(-500, 500, step=10))
    ax2.axis('equal')
    ax2.grid()
    if if_on_screen:
        plt.ion()
    else:
        plt.ioff()

In [ ]:
# for all frames do
if_replace = False
errorTransSum = 0
errorRotSum = 0
errorRot_list = []
errorTrans_list =[]

for frame in range(first_frame, last_frame):
    # 1-based index
    k = frame-first_frame+1

    # read current images
    I = imread(os.path.join(img_dir, '%06d.png'%frame))
    assert(len(I.shape) == 2) # should be grayscale

    # compute egomotion
    process_result = visoMono.process_frame(I, if_replace)
    Tr = visoMono.getMotion()
    matrixer = viso2.Matrix(Tr)
    Tr_np = np.zeros((4, 4))
    Tr.toNumpy(Tr_np) # so awkward...

    # accumulate egomotion, starting with second frame
    if k > 1:
        if process_result is False:
            if_replace = True
            Tr_total.append(Tr_total[-1])
            Tr_total_np.append(Tr_total_np[-1])
        else:
            if_replace = False
            Tr_total.append(Tr_total[-1] * viso2.Matrix.inv(Tr))
            Tr_total_np.append(Tr_total_np[-1] @ np.linalg.inv(Tr_np)) # should be the same
            print(Tr_total_np[-1])

    # output statistics
    num_matches = visoMono.getNumberOfMatches()
    num_inliers = visoMono.getNumberOfInliers()
    matches = visoMono.getMatches()
    matches_np = np.empty([4, matches.size()])

    for i,m in enumerate(matches):
        matches_np[:, i] = (m.u1p, m.v1p, m.u1c, m.v1c)

    if if_vis:
        # update image
        ax1.clear()
        ax1.imshow(I, cmap='gray', vmin=0, vmax=255)
        if num_matches != 0:
            for n in range(num_matches):
                ax1.plot([matches_np[0, n], matches_np[2, n]], [matches_np[1, n], matches_np[3, n]])
        ax1.set_title('Frame %d'%frame)

        # update trajectory
        if k > 1:
            ax2.plot([Tr_total_np[k-2][0, 3], Tr_total_np[k-1][0, 3]], \
                [Tr_total_np[k-2][2, 3], Tr_total_np[k-1][2, 3]], 'b.-', linewidth=1)
            ax2.plot([gtTr[k-2][0, 3], gtTr[k-1][0, 3]], \
                [gtTr[k-2][2, 3], gtTr[k-1][2, 3]], 'r.-', linewidth=1)
        ax2.set_title('Blue: estimated trajectory; Red: ground truth trejectory')

        plt.draw()

    # Compute rotation
    Rpred_p = Tr_total_np[k-2][0:3, 0:3]
    Rpred_c = Tr_total_np[k-1][0:3, 0:3]
    Rpred = Rpred_c.transpose() @ Rpred_p
    Rgt_p = np.squeeze(gtTr[k-2, 0:3, 0:3])
    Rgt_c = np.squeeze(gtTr[k-1, 0:3, 0:3])
    Rgt = Rgt_c.transpose() @ Rgt_p

    # Compute translation
    Tpred_p = Tr_total_np[k-2][0:3, 3:4]
    Tpred_c = Tr_total_np[k-1][0:3, 3:4]
    Tpred = Tpred_c - Tpred_p
    Tgt_p = gtTr[k-2, 0:3, 3:4]
    Tgt_c = gtTr[k-1, 0:3, 3:4]
    Tgt = Tgt_c - Tgt_p

    # Compute errors
    errorRot, errorTrans = errorMetric(Rpred, Rgt, Tpred, Tgt)
    errorRotSum = errorRotSum + errorRot
    errorTransSum = errorTransSum + errorTrans
    # errorRot_list.append(errorRot)
    # errorTrans_list.append(errorTrans)

    print('Mean Error Rotation: %.5f'%(errorRotSum / (k-1+epi)))
    print('Mean Error Translation: %.5f'%(errorTransSum / (k-1+epi)))

    print('== [Result] Frame: %d, Matches %d, Inliers: %.2f'%(frame, num_matches, 100*num_inliers/(num_matches+1e-8)))

    if if_vis:
        # input('Paused; Press Enter to continue') # Option 1: Manually pause and resume
        if if_on_screen:
            plt.pause(0.1) # Or Option 2: enable to this to auto pause for a while after daring to enable animation in case of a delay in drawing
        vis_path = os.path.join(save_path, 'frame%03d.jpg'%frame)
        fig.savefig(vis_path)
        print('Saved at %s'%vis_path)
        
        if frame % 50 == 0 or frame == last_frame-1:
            plt.figure(figsize=(10, 15))
            plt.imshow(plt.imread(vis_path))
            plt.axis('off')
            plt.show()


# input('Press Enter to exit')

1. Report the final rotation and translation error. **(2 points)**

``report two error metrics over all frames here``

Then answer the questions below

2. In $\mathtt{libviso2}$, the feature points are "bucketed" ($\mathtt{pyviso/src/matcher.cpp: Line 285 - 326}$), which means in a certain area of region, the number of detected keypoint pairs should be within certain bounds. Why?  **(3 points)**

``answer here``

3. We have run SFM on a single camera, which means the scale of translation is unknown. However, as you may have observed, the predicted trajectory is still somehow similar to the ground-truth trajectory. How does $\mathtt{libviso2}$ handle this ambiguity ($\mathtt{viso\_mono.cpp: Line 245}$)?  **(5 points)**

``answer here``

4. Briefly explain the RANSAC algorithm used in $\mathtt{libviso2}$ ($\mathtt{viso\_mono.cpp: Line 113 - 129}$).  **(5 points)**

``answer here``

## Question 2: Using SIFT [4] for SFM

In the second task, you are required to use keypoints and feature descriptors from SIFT for SFM. The SIFT implementation can be found in directory $\mathtt{SIFT}$. 

(A) Go to $\mathtt{./pyviso/SIFT}$ directory and run $\mathtt{runSIFT.py}$ (e.g. `python runSIFT.py --input ../../dataset_SfM/sequences/00/image_0/`). You will save the detected keypoints and feature descriptors under the directory $\mathtt{SIFT}$. For image $\mathtt{000abc.png}$, the pre-computed features and keypoints should be saved in a $\mathtt{.npy}$ file named as $\mathtt{pyviso/SIFT/000abc\_feature.npy}$. The variable should be a $130 \times N$ matrix with $\mathtt{single}$ precision, where $N$ is the number of feature points being detected. For each $130$-dimensional feature vector, the first two dimensions are the location of the keypoints (column number first and then row number) on the image plane and the last $128$ dimensions are the feature descriptor. 

If you enabled visualization, the visualized correspondences and trajectory will be saved to $\mathtt{vis\_preFeature/frame\{\}.jpg}$.

(B) Run the following script

In [ ]:
import pyviso.runFeature as runFeature
dataset_path = './dataset_SfM'
feature_dir = 'pyviso/SIFT'

runFeature.runSFM(dataset_path, feature_dir )

1. Report the final rotation and translation error. **(2 points)**

``report two error metrics over all frames here``

Next, answer the following questions:

2. Does SIFT yield higher accuracy than the original $\mathtt{libviso2}$? Why or why not? If not, can you suggest one possible way to improve? **(5 points)**

``answer here``

3. Explain how SIFT achieves invariance to 
       a. illumination
       b. rotation
       c. scale
 **(3 points)**

``answer here``

## Question 3: Using SPyNet [5] for SFM

Now we will compute camera motion from optical flow computed using SPyNet. We first uniformly sample points in an image, then consider the flow-displaced point in the other image as a match. A modified PyTorch implementation of SPyNet is provided in directory  $\mathtt{Flow}$.

(A) Go to $\mathtt{pyviso/Flow}$ and run $\mathtt{demo\_spynet.py}$ (e.g. `python demo_spynet.py --input ../../dataset_SfM/sequences/00/image_0`). Try starting with processing only one sample via `--last_frame 1`. If it works fine, remove it and run inference over all images.

(B) Run the following script.

In [ ]:
import pyviso.runMatch as runMatch
dataset_path = './dataset_SfM'
feature_dir = 'pyviso/Flow'
runMatch.runSFM(dataset_path, feature_dir )

1. Report the final rotation and translation error. **(2 points)**

``report two error metrics over all frames here``

Next, answer the following questions:

2. Does SPyNet yield higher accuracy than the original $\mathtt{libviso2}$? Why or why not? If not, can you suggest one possible step to improve? **(5 points)**

``answer here``

3. Explain how SPyNet achieves accurate flow with significantly lower computational cost. **(3 points)**

``answer here``

## Question 4: Optical Flow with Lucas-Kanade (LK) [7]

In this question you are going to implement the Lucas-Kanade algorithm and test it on a image sequence.

1. Implement the basic Lucas-Kanade method for estimating optical flow. The functions ``def extract_window`` and ``def LucasKanadeOpticalFlow`` need to be completed and they can then be used for the main script in the second block (only the code corresponding to LKmode == 'none' is needed for questions 4.1 to 4.3).  **(5 points)**

``answer in the code block below``

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def grayscale(img):
    '''
    Converts RGB image to Grayscale
    '''
    gray=np.zeros((img.shape[0],img.shape[1]))
    gray=img[:,:,0]*0.2989+img[:,:,1]*0.5870+img[:,:,2]*0.1140
    return gray

def plot_optical_flow(img,U,V,titleStr, if_gray_scale=False):
    '''
    Plots optical flow given U,V and one of the images
    '''
    
    # Change t if required, affects the number of arrows
    # t should be between 1 and min(U.shape[0],U.shape[1])
    t = 10 
    
    # Subsample U and V to get visually pleasing output
    U1 = U[::t,::t]
    V1 = V[::t,::t]
    
    # Create meshgrid of subsampled coordinates
    r, c = img.shape[0],img.shape[1]
    cols,rows = np.meshgrid(np.linspace(0,c-1,c), np.linspace(0,r-1,r))
    cols = cols[::t,::t]
    rows = rows[::t,::t]
    
    # Plot optical flow
    plt.figure(figsize=(10,10))
    if if_gray_scale:
        plt.imshow((img*255.).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
    else:
        plt.imshow(img)
    plt.quiver(cols,rows,U1,V1)
    plt.title(titleStr)
    plt.show()

def extract_window(img, x, y, window_size):
    '''
    The functions takes a grayscale image, a center pixel (x, y), and a half window size, 
    outputs a patch of the images of size [window_size*2+1, window_size*2+1]
    '''
    ######## (Q4.1) YOUR CODE HERE #######
    ## x1 = 
    ## x2 =
    ## y1 =
    ## y2 =     
    ##########################
    
    window = img[x1 : x2, y1 : y2]
    assert window.shape == (window_size*2+1, window_size*2+1)
    coords = (x1, x2, y1, y2)
    
    return window

    
def LucasKanadeOpticalFlow(I1g, I2g, window_size, u_prev=None, v_prev=None, LKmode='none'):
    '''
    Implement the Lucas-Kanade algorithm
    Inputs: the two images, window size
    Returns: u, v - the optical flow
    '''
    
    import numpy as np
    from scipy import signal
    
    kernel_x = np.array([[-1., 1.], [-1., 1.]])
    kernel_y = np.array([[1., 1.], [-1., -1.]])
    kernel_t = np.array([[1., 1.], [1., 1.]]) *.25
    w = int(np.ceil(window_size/2)) # window_size is odd, all the pixels with offset in between [-w, w] are inside the window
    I1g = I1g / 255. # normalize pixels
    I2g = I2g / 255. # normalize pixels
    
    # Implement Lucas Kanade for each point, calculate I_x, I_y, I_t
    mode = 'same'
    fx = signal.convolve2d(I1g, kernel_x, boundary='symm', mode=mode)
    fy = signal.convolve2d(I1g, kernel_y, boundary='symm', mode=mode)
    ft = signal.convolve2d(I2g, kernel_t, boundary='symm', mode=mode) + \
         signal.convolve2d(I1g, -kernel_t, boundary='symm', mode=mode) # I_t with Gaussian Smoothing
    #ft = I2g - I1g # Vanilla I_t without Gaussian Smoothing

    u = np.zeros(I1g.shape)
    v = np.zeros(I1g.shape)
    
    # within window window_size * window_size
    for i in range(w, I1g.shape[0]-w):
        for j in range(w, I1g.shape[1]-w):    
            Ix = extract_window(fx, i, j, w).flatten()
            
            if LKmode == 'none':
                Iy = extract_window(fy, i, j, w).flatten()
                
            elif LKmode == 'multiscale':
                assert u_prev is not None and v_prev is not None
                assert v_prev.shape == I1g.shape
                '''
                Use i, and u_prev or v_prev to get the accumulated flow i_accu in this scale
                Use j, and u_prev or v_prev to get the accumulated flow j_accu in this scale
                Make sure i_accu, j_accu does not go out of bound (i.e. i in range(w, I1g.shape[0]-w), j in range(w, I1g.shape[1]-w)); a simple np.clip will do the trick
                '''
                i_accu = # (Q4.4) YOUR CODE HERE
                j_accu = # (Q4.4) YOUR CODE HERE
                Iy = extract_window(fy, i_accu, j_accu, w).flatten()
                
            else:
                raise (RuntimeError('Invalid LK mode!'))
                
            It = extract_window(ft, i, j, w).flatten()
            
            b = ... # (Q4.4) YOUR CODE HERE, based on the math on slide #45 of Lecture 5
            A = ... # (Q4.4) YOUR CODE HERE, based on the math on slide #45 of Lecture 5

            ws, _ = np.linalg.eig(A.transpose() @ A)
            lam = 1e-5
            tau= 1e-6

            # if threshold τ is larger than the smallest eigenvalue of A'A:
            if ws[1] > tau:
                nu =  # (Q4.1) YOUR CODE HERE: get velocity nu based on (A^T A + lam * np.eye(2)) nu = A^T b
                u[i,j]=nu[0]
                v[i,j]=nu[1]
            else:
                u[i,j]=0.
                v[i,j]=0.
    
    return u, v

2. Plot standard Lucas-Kanade optical flow (that is, with LKmode as 'none') for the three choices for the pair of images im1 and im2 specified below, using window sizes of 13. For each plot, comment on whether and how the estimated flow matches your intuition given the motion visible for the image pair.  **(6 points)**

In [ ]:
images = []
for i in range(1,5):
    images.append(plt.imread('OpticalFlowImages/im'+str(i)+'.png')[:,:288,:])
# each image after converting to gray scale is of size -> 400x288

window = 13
U, V = LucasKanadeOpticalFlow(grayscale(images[0]),grayscale(images[1]), window)
plot_optical_flow(images[0], U, V, 'image2 = ' + str(1) + ', window = ' + str(window))

window = 13
U, V = LucasKanadeOpticalFlow(grayscale(images[0]),grayscale(images[2]), window)
plot_optical_flow(images[0], U, V, 'image2 = ' + str(2) + ', window = ' + str(window))

window = 13
U, V = LucasKanadeOpticalFlow(grayscale(images[0]),grayscale(images[3]), window)
plot_optical_flow(images[0], U, V, 'image2 = ' + str(3) + ', window = ' + str(window))

``display your plots above by fully expanding the above block, or your plots here``

``comment here``

3. For the first pair of images above, estimate two new optical flows using one smaller-sized window and another larger-sized window, which lead to observable difference in the results. Comment on the effect of window size on the results and justify.  **(4 points)**

``copy your results here``

``answer and comment here``

4. Implement the incomplete lines within `def LucasKanadeOpticalFlow` in the first block to handle accumulated flow in the coarse-to-fine scheme (corresponding to LKmode == 'multiscale'). Then complete the `def opticalFlowMultiScale` function below and run the block to compute and visualize flow in coarse-to-fine scales. **(5 points)**

One of your output images which is at the coarest scale should look like this (but doesn't have to be identical):
![](images/LK.png)

In [ ]:
from scipy.interpolate import interp2d

def upsampleFlow(U_prev, V_prev):
    x, y = U_prev.shape
    x_new = np.linspace(0, x, x*2)
    y_new = np.linspace(0, y, y*2)
    
    func_u = interp2d(np.array(np.arange(y)), np.array(np.arange(x)), U_prev)
    u = func_u(y_new, x_new) * 2
    
    func_v = interp2d(np.array(np.arange(y)), np.array(np.arange(x)), V_prev)
    v = func_v(y_new, x_new) * 2
    
    return u, v

def opticalFlowMultiScale(im1, im2, window_size, num_levels):
    im1_arr, im2_arr = [], []
    u_list, v_list = [], []
    for i in range(num_levels):
        im1_arr.insert(0, im1)
        im2_arr.insert(0, im2)
        im1 = im1[::2 , ::2]
        im2 = im2[::2 , ::2]
    
    print("Level 0: image_shape: ", im1_arr[0].shape)
    u, v = LucasKanadeOpticalFlow( # YOUR CODE HERE: compute flow in the coarest scale
    u_list.append(u)
    v_list.append(v)
    
    for i in range(1, num_levels):
        print("Level {}: image_shape: {}".format(i, im1_arr[i].shape))
        ######## (Q4.4) YOUR CODE HERE #######
        # upsample previous flow
        # use the image pair from current level i, along with upsampled flow as inputs
        ##########################
        u_list.append(u_i)
        v_list.append(v_i)

    return u_list, v_list, im1_arr, im2_arr

images = []
for i in range(1,5):
    images.append(plt.imread('OpticalFlowImages/im'+str(i)+'.png')[:,:288,:])
# each image after converting to gray scale is of size -> 400x288

window = 13
U_list, V_list, im1_arr, im2_arr = opticalFlowMultiScale(grayscale(images[0]),grayscale(images[3]), window, 3)

for l, (U, V, im1) in enumerate(zip(U_list, V_list, im1_arr)):
    print(U.shape, im1.shape)
    plot_optical_flow(im1, U, V, 'level = ' + str(l) + ' image2 = ' + str(3) + ', window = ' + str(window), if_gray_scale=True)

5. We will now consider two pairs of KITTI images located at `/datasets/cs152b-sp22-a00-public/dataset_SfM/sequences/00/image_0`, from a video sequence recorded while driving down a street.

First consider the image pair 000000.png and 000001.png shown below. Compute the coarse-to-fine optical flow and plot the output at the finest scale.
![](images/kitti.png)

Next, consider the image pair 000106.png and 000107.png shown below. Compute the coarse-to-fine optical flow and plot the output at the finest scale.
![](images/kittiRE.png)

Now describe the pattern of flow vectors in the above two examples and explain how each of them relates to the type of camera motion between each pair of images.  **(6 points)**

``your optical flow output here for the first pair of images``

``your optical flow output here for the second pair of images``

``explain flow patterns and their relation to observed motions``

6. Now, use a camera to take two images of your own with very small motion (for example, consecutive frames of a video sequence). Compute the coarse-to-fine optical flow and comment on whether the flow vectors are consistent with what you expected given the motion. **(4 points)**

You may need to resize the two images to the same size, where both height and width are divisible by 4 (as required by the coarse-to-fine code). Feel free to adjust the window size and number of scales to achived better results. 

``your plots here``

``comment here``

# References
1. Daniel DeTone, Tomasz Malisiewicz, and Andrew Rabinovich. Superpoint: Self-supervised interest point detection and description. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition Workshops, pages 224–236, 2018.
2. Andreas Geiger, Philip Lenz, and Raquel Urtasun. Are we ready for autonomous driving? the kitti vision benchmark suite. In Conference on Computer Vision and Pattern Recognition (CVPR), 2012.
3. Andreas Geiger, Julius Ziegler, and Christoph Stiller. Stereoscan: Dense 3d reconstruction in real-time. In Intelligent Vehicles Symposium (IV), 2011.
4. David G Lowe. Distinctive image features from scale-invariant keypoints. IJCV, 60(2):91–110, 2004.
5. Anurag Ranjan and Michael J Black. Optical flow estimation using a spatial pyramid network. In
Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition, pages 4161–4170, 2017.
6. A. Vedaldi and B. Fulkerson. VLFeat: An open and portable library of computer vision algorithms. http://www.vlfeat.org/, 2008.
7. Lucas, Bruce D., and Takeo Kanade. "An iterative image registration technique with an application to stereo vision." (1981): 674.